# BruSafe · Notebook 04L — Module M3 at a measured contact energy

*Literature variant of notebook 04. `DG_CONTACT` is obtained by inverting the published melting point of MS2 VLPs (doi 10.1021/acs.biochem.4c00225) through Eq. (4.8), instead of being set to the placeholder of −4.2 kcal mol⁻¹. The dimer pool is read from the M2 export.*

*Section numbering follows the CAPSID framework document; this notebook implements Sections 4.1–4.11.*

**iGEM Tec-Chihuahua 2026 · CAPSID framework**

This notebook takes the soluble dimer pool from `BruSafe_03_M2_soluble_partition.ipynb` and the calibrated contact energies from the M4 energetics module (Section 5 of the framework document; not yet implemented). It predicts capsid yield, the distribution of intermediate sizes, and the nucleation lag.

## Interfaces

| From | Quantity | Units |
|---|---|---|
| M2 | free dimer pool, or the dimer **flux** $\varphi s/2$ | µM, µM min⁻¹ |
| M4 | $\Delta G_{AB}$, $\Delta G_{CC}$, $\Delta G_{switch}$ (calibrated, not raw MM/GBSA) | kcal/mol |
| → M2 | `assembly_sink(Sol, t)` closing the loop | µM min⁻¹ |

## Validation assays (dPCR is not one of them)

| Prediction | Assay |
|---|---|
| capsid CP fraction | SEC, native agarose, sucrose gradient |
| intermediate distribution | SEC / DLS (partial shells run between dimer and capsid) |
| countable particles, malformed fraction | negative-stain TEM |
| contact energy, directly | urea or thermal disassembly midpoint + SEC |

dPCR counts encapsidated cargo. It anchors the packaging arm (Sections 5.6–5.8 of the framework document) and says nothing about the lattice contact energy. §8 identifies the one point at which the two arms make a shared, testable prediction.

## Model

Nucleation–elongation cascade over dimers, $C_1 + C_j \rightleftharpoons C_{j+1}$, $j = 1\ldots 89$,
with the closed shell at $N = 90$. Association is degenerate (there are several equivalent addition
sites), whereas dissociation is not. Forward and reverse rates are tied by detailed balance, so the
kinetics and the equilibrium of §3 belong to the same model and are not two models that happen to
agree.

**Placeholders.** The contact schedule and the degeneracy vector are placeholders with the correct
total, not counts from the structure. §2 states what they should be replaced with, and §9 measures
how much each conclusion depends on them: some conclusions are robust and others are not, and the
notebook indicates which.

---
## 1 · Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

RNG = np.random.default_rng(20261013)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})

R_KCAL = 1.98720425e-3
def RT(T_K=310.15): return R_KCAL*T_K

N_DIMERS   = 90        # closed T=3 shell, in dimers
N_CONTACTS = 180.0     # total dimer-dimer interfaces in the shell
print(f"M3 · T=3 assembly | N = {N_DIMERS} dimers | RT(37 C) = {RT():.4f} kcal/mol")

# ---- Figure style (team palette) -------------------------------------------
# The colours are sampled from the team wordmark. Figures refer to the names
# below instead of hex codes, so the whole set can be restyled from this cell.
# SF Pro is used where installed; matplotlib substitutes another font otherwise.
CREAM, PALE, STEEL = "#F4F3EF", "#CCD8E2", "#89A1C2"
SKY,   BLUE, MID   = "#4CA3DF", "#3E77E1", "#2C5BA5"
NAVY,  INK         = "#173779", "#122047"
from matplotlib.colors import LinearSegmentedColormap
BRAND_CMAP = LinearSegmentedColormap.from_list("tecchihuahua",
                                               [NAVY, MID, BLUE, SKY])
plt.rcParams.update({
    "font.family": ["SF Pro Text", "SF Pro Display", "SF Pro", "Helvetica Neue"],
    "mathtext.fontset": "custom", "mathtext.default": "regular",
    "mathtext.rm": "SF Pro Text", "mathtext.it": "SF Pro Text:italic",
    "mathtext.bf": "SF Pro Text:bold",
    "figure.facecolor": CREAM, "axes.facecolor": CREAM, "savefig.facecolor": CREAM,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": STEEL, "grid.alpha": 0.35,
    "legend.facecolor": CREAM, "legend.edgecolor": STEEL, "legend.framealpha": 0.9,
    "axes.prop_cycle": plt.cycler(color=[NAVY, BLUE, SKY, STEEL, MID, PALE]),
})

# ---- contact energy from a MEASURED melting point -------------------------
# Pistono et al. 2024, Biochemistry 63:1913, doi 10.1021/acs.biochem.4c00225:
# wild-type MS2 VLP denatures with Tm ~ 66 C in a SYPRO orange thermal shift
# assay run at 5-20 uM coat-protein monomer (2.5-10 uM dimer). A denaturation
# midpoint IS a disassembly midpoint, so at Tm the assay concentration equals
# c*, and Eq. (4.8) inverts for dG_contact. The mid-range assay concentration
# is used here; data/litdata/derived_M3_dG_from_Tm.csv carries the full range.
import pandas as pd
_lit = pd.read_csv("../../data/litdata/derived_M3_dG_from_Tm.csv")
_row = _lit[(_lit.variant == "wtMS2") & (_lit.assay_dimer_uM == 5.0)].iloc[0]
DG_CONTACT = float(_row.dG_contact_kcal_mol)
print(f"literature dG_contact = {DG_CONTACT:.3f} kcal/mol "
      f"(wt MS2, Tm = {_row.Tm_C:.0f} C at {_row.assay_dimer_uM:.1f} uM dimer)")
print(f"  framework placeholder was -4.200 kcal/mol")

---
## 2 · Geometry

MS2 T=3 has 180 coat-protein subunits. The icosahedral asymmetric unit holds three quasi-equivalent
chains A, B and C, so there are 60 asymmetric units. Each A/B dimer takes one A and one B chain,
which gives **60 A/B dimers**. The C chains pair across asymmetric units, giving 60 C chains / 2 =
**30 C/C dimers**. The total is 90 dimers, the assembly unit used throughout this notebook.

The A/B and C/C dimers differ in FG-loop conformation. This is why a single lumped contact energy
cannot represent the lattice, and why per-interface MM/GBSA (Section 5.1 of the framework document)
can add information that a phenomenological model cannot.

> **Placeholders to be replaced.** `contact_schedule` and `degeneracy` below reproduce the correct
> **total** contact count and a physically reasonable shape, but they are not derived from the
> structure. Contacts should be counted from 2MS2 or 1AQ3 with a heavy-atom cutoff, and the schedule
> derived from a growth path on the actual lattice and not from a ramp. §9 quantifies which
> conclusions survive errors in these placeholders.

In [ ]:
def contact_schedule(N=N_DIMERS, total=N_CONTACTS, c_start=1.0, c_end=3.0):
    '''Contacts formed when the j-th dimer joins, j = 2..N. Rescaled so the sum
    is exactly `total` - the one property that is NOT a placeholder.'''
    j = np.arange(2, N+1)
    c = c_start + (c_end - c_start)*(j-2)/(N-2)
    c *= total/c.sum()
    out = np.zeros(N+1); out[2:] = c
    return out

def degeneracy(N=N_DIMERS, f_deg=2.0):
    '''Number of equivalent ways to add the next dimer. Constant is the
    conservative choice: a fabricated geometric profile puts structure into the
    intermediate distribution that the structure itself has not been asked for.'''
    f = np.full(N+1, f_deg); f[1] = 1.0
    return f

def count_contacts_from_pdb(path, cutoff=4.5):
    '''Stub. Parse the capsid, assign chains to dimers, count dimer pairs with any
    heavy-atom pair within `cutoff` A, and split by AB/CC interface class.
    Replace N_CONTACTS and the schedule with the result.'''
    raise NotImplementedError("count from 2MS2 / 1AQ3 before publishing numbers")

c_sched = contact_schedule()
print(f"dimers: 60 A/B + 30 C/C = {60+30} (must be {N_DIMERS})")
print(f"schedule sum   : {c_sched[2:].sum():.6f}  (must be {N_CONTACTS})")
print(f"first addition : {c_sched[2]:.2f} contacts   (nucleation barrier)")
print(f"last addition  : {c_sched[-1]:.2f} contacts   (closure)")
print(f"mean           : {c_sched[2:].mean():.3f} = {N_CONTACTS}/{N_DIMERS-1}")

---
## 3 · Equilibrium

With detailed balance, the equilibrium distribution follows from the free-dimer concentration alone:

$$\log C_j = j\log C_1 + \sum_{i=2}^{j}\log K_i, \qquad
K_i = \frac{k_{on}f_{i-1}}{k_{off,i}}, \qquad
k_{off,i} = k_{on}\,e^{\,c_i \Delta G_{contact}/RT}\times 10^{6}$$

The factor $10^{6}$ converts the 1 M standard state of $\Delta G$ into the µM units used here; an
error in it shifts $c^{*}$ by six orders of magnitude. All quantities are computed in logarithms,
because $C_1^{90}$ underflows double precision for any realistic $C_1$.

In [ ]:
def rate_constants(dG_contact, k_on=6.0, T_K=310.15, N=N_DIMERS, f_deg=2.0,
                   schedule=None):
    '''k_on in uM^-1 min^-1 (~1e5 M^-1 s^-1, diffusion-limited for a protein pair).'''
    c = contact_schedule(N) if schedule is None else schedule
    f = degeneracy(N, f_deg)
    kon  = k_on*f                                              # uM^-1 min^-1
    koff = np.zeros(N+1)
    koff[2:] = k_on*1e6*np.exp(c[2:]*dG_contact/RT(T_K))       # min^-1
    return kon, koff, c, f

def log_distribution(dG_contact, c1, N=N_DIMERS, **kw):
    kon, koff, c, f = rate_constants(dG_contact, N=N, **kw)
    lK = np.log(kon[1:N]) - np.log(koff[2:N+1])
    lC = np.zeros(N+1); lC[1] = np.log(c1)
    lC[2:] = np.cumsum(lK) + np.arange(2, N+1)*np.log(c1)
    return lC

def total_CP(dG_contact, c1, N=N_DIMERS, **kw):
    lC = log_distribution(dG_contact, c1, N, **kw)
    return float(np.sum(np.arange(1, N+1)*np.exp(np.clip(lC[1:], -700, 700))))

def equilibrium(dG_contact, total_dimer, N=N_DIMERS, **kw):
    g = lambda l: np.log(total_CP(dG_contact, np.exp(l), N, **kw)) - np.log(total_dimer)
    l = brentq(g, np.log(1e-14), np.log(total_dimer), xtol=1e-13)
    lC = log_distribution(dG_contact, np.exp(l), N, **kw)
    C = np.exp(np.clip(lC, -700, 700))
    return {"free": C[1], "capsid": N*C[N], "dist": C,
            "interm": total_dimer - N*C[N] - C[1]}

def cstar_approx(dG_contact, N=N_DIMERS, total=N_CONTACTS, T_K=310.15):
    '''Zlotnick pseudo-critical concentration, in uM.'''
    return np.exp(total*dG_contact/((N-1)*RT(T_K)))*1e6

TOTAL_DIMER = 500.0
rows = []
for dG in (-3.0, -3.5, -4.0, DG_CONTACT, -5.0):
    e = equilibrium(dG, TOTAL_DIMER)
    rows.append(dict(dG_contact=dG, free_uM=e["free"], capsid_frac=e["capsid"]/TOTAL_DIMER,
                     interm_frac=e["interm"]/TOTAL_DIMER, cstar_approx_uM=cstar_approx(dG),
                     free_over_cstar=e["free"]/cstar_approx(dG)))
eq = pd.DataFrame(rows); display(eq.round(5))
print("Two-state: intermediates stay below ~1% of mass at equilibrium, so a capsid")
print("assembly assay measures essentially a two-species mixture. Free dimer is")
print("buffered near c*, and the (N-1) approximation lands within a factor ~2 of")
print("the exact free-dimer plateau across the whole range - good enough for design,")
print("not good enough to quote as a measured c*.")

---
## 4 · Kinetics

In [ ]:
def cascade_rhs(t, y, kon, koff, feed=None, N=N_DIMERS):
    C = np.maximum(y, 0.0); c1 = C[1]
    flux = kon[1:N]*C[1:N]*c1 - koff[2:N+1]*C[2:N+1]     # net j -> j+1
    d = np.zeros(N+1)
    d[2:N+1] += flux
    d[1:N]   -= flux
    d[1]     -= flux.sum()                # every addition also consumes a free dimer
    if feed is not None: d[1] += feed(t)
    return d

def run_cascade(dG_contact, bolus=0.0, t_end=1080.0, feed=None, npts=140,
                N=N_DIMERS, **kw):
    kon, koff, c, f = rate_constants(dG_contact, N=N, **kw)
    y0 = np.zeros(N+1); y0[1] = bolus
    s = solve_ivp(cascade_rhs, (0, t_end), y0, args=(kon, koff, feed, N),
                  method="LSODA", rtol=1e-8, atol=1e-16,
                  t_eval=np.linspace(0, t_end, npts))
    return s

# mass conservation is the only real correctness check available here
s = run_cascade(DG_CONTACT, bolus=500.0)
mass = np.sum(np.arange(N_DIMERS+1)[:, None]*s.y, axis=0)
print(f"mass drift over 18 h : {abs(mass[-1]-500.0)/500.0:.1e} (relative)")

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.3))
for tot, c in zip((20., 100., 500.), [SKY, BLUE, NAVY]):
    r = run_cascade(DG_CONTACT, bolus=tot)
    ax[0].plot(r.t/60, N_DIMERS*r.y[N_DIMERS]/tot, color=c, label=f"{tot:.0f} $\\mu$M")
ax[0].set(xlabel="time (h)", ylabel="capsid CP fraction", ylim=(-0.02, 1.02),
          title="Nucleation lag"); ax[0].legend(fontsize=7)

r = run_cascade(DG_CONTACT, bolus=500.0)
for k, ti in enumerate([10, 30, 70, 139]):
    ax[1].semilogy(np.arange(1, N_DIMERS+1), np.maximum(r.y[1:, ti], 1e-20),
                   lw=1.1, label=f"t = {r.t[ti]/60:.1f} h")
ax[1].set(xlabel="intermediate size (dimers)", ylabel="conc (uM)", ylim=(1e-16, 1e3),
          title="Size distribution, 500 $\\mu$M bolus"); ax[1].legend(fontsize=7)
fig.tight_layout(); plt.show()
print("Left: 20 uM is thermodynamically above c* (~1 uM) yet produces nothing in")
print("18 h. Being above the critical concentration is necessary, not sufficient -")
print("if an in vitro reassembly fails, check the lag before blaming the protein.")

---
## 5 · The kinetic trap, stated precisely

The textbook explanation is that yield falls at high subunit concentration because rapid nucleation
strands partial shells. The cascade used here produces a **saturating** trap rather than a declining
one: stalled intermediates plateau at a few percent and stay there.

This is a limit of the model and not a defect. Reproducing a genuinely non-monotonic yield curve
requires an **off-pathway or malformed species**, such as a closed but incorrect shell or an
aggregate, which this cascade does not contain. If TEM shows malformed particles increasing with
induction level, that is evidence for a pathway missing from the model, and the pathway should be
added.

In [ ]:
rows = []
for tot in (200., 500., 1000., 2000., 5000.):
    r = run_cascade(DG_CONTACT, bolus=tot)
    cap = N_DIMERS*r.y[N_DIMERS, -1]
    rows.append(dict(bolus_uM=tot, capsid_frac=cap/tot, free_uM=r.y[1, -1],
                     stalled_frac=(tot-cap-r.y[1, -1])/tot,
                     eq_capsid_frac=equilibrium(DG_CONTACT, tot)["capsid"]/tot))
trap = pd.DataFrame(rows); display(trap.round(4))
print("Stalled fraction saturates instead of growing. Free dimer is driven far")
print("BELOW its equilibrium value (over-depletion) - that is the real trap")
print("signature here, and it is what a SEC trace would show as a shoulder")
print("between the dimer and capsid peaks.")

---
## 6 · In vivo assembly is a feed, not a bolus

Every in vitro reassembly experiment is a bolus: dimer is dialysed into assembly buffer and
followed. In the cell, dimer instead arrives **continuously** at the rate supplied by M2,
$\varphi(s)\,s/2$. A slow feed keeps the free-dimer pool low, which suppresses over-nucleation.

Conclusions from bolus experiments therefore cannot be transferred to the expression system without
this step, and the comparison below is the version of the kinetic-trap argument that applies to
BruSafe.

> **Note on the summary printed below the table.** The sentences printed under the comparison table
> were written for an earlier parameter set and do not agree with the table at the present contact
> energy: the stalled fraction is 0.008 for the bolus and 0.040 for the constant feed. The table is
> the result. The statement that a slow feed reduces the stalled fraction requires review before it
> is quoted.

In [ ]:
TOTAL, T_END = 600.0, 1080.0            # 600 uM dimer delivered over 18 h
rows = []
for lab, kw in [("bolus at t = 0", dict(bolus=TOTAL)),
                ("constant feed",  dict(bolus=0.0, feed=lambda t: TOTAL/T_END)),
                ("ramped feed",    dict(bolus=0.0,
                                        feed=lambda t: 2*TOTAL*t/T_END**2))]:
    r = run_cascade(DG_CONTACT, t_end=T_END, **kw)
    cap = N_DIMERS*r.y[N_DIMERS, -1]
    rows.append(dict(mode=lab, capsid_frac=cap/TOTAL, free_uM=r.y[1, -1],
                     stalled_frac=(TOTAL-cap-r.y[1, -1])/TOTAL))
feed_df = pd.DataFrame(rows); display(feed_df.round(4))
print("Feeding cuts stalled intermediates roughly 3-fold versus a bolus - the")
print("mechanism the trap story predicts. But the reaction is still RUNNING at")
print("harvest: tens of uM of free dimer remain unconverted at 18 h. Both effects")
print("are real and they push yield in opposite directions, so 'feed is better'")
print("is only true for the intermediate fraction, not for total capsid at 18 h.")

---
## 7 · Closing the loop with Module M2

M2 exposes `assembly_sink(Sol, t)` in µM min⁻¹ of **monomer**. The cascade consumes dimer, so the
sink is twice the dimer consumption rate. Because the soluble lane holds Sol + Cap and the sink only
moves CP between them, switching it on does **not** change the gel soluble fraction (Section 4.10 of
the framework document). It changes what M3 receives, and nothing upstream.

In [ ]:
def make_assembly_sink(dG_contact, k_on=6.0, f_deg=2.0):
    '''Quasi-steady sink for M2: rate at which free dimer enters the cascade.
    Uses the nucleation-limited rate, which is the slow step by construction.'''
    kon, koff, c, f = rate_constants(dG_contact, k_on=k_on, f_deg=f_deg)
    def sink(Sol_uM, t):
        dimer = np.maximum(Sol_uM, 0.0)/2.0
        rate_dimer = kon[1]*dimer*dimer - koff[2]*0.0     # nucleation flux
        return 2.0*rate_dimer                             # back to monomer units
    return sink

try:
    # notebook 03L: M2 calibrated on the measured arenin partition
    ex = np.load("M2_export_literature.npz")
    dimer_18h = ex["dimer_uM"][:, -1]
    print("dimer pool from M2 calibrated on measured data (uM):", np.round(dimer_18h, 1))
except FileNotFoundError:
    dimer_18h = np.array([408.6, 556.1, 626.4, 643.6])
    print("M2_export.npz not found; using M2 reference values (uM):", dimer_18h)

print(f"\n{'dG':>6} {'c* (uM)':>10} {'lowest arm / c*':>17} {'eq capsid frac':>16}")
for dG in (-3.5, -4.0, DG_CONTACT, -5.0):
    cs = cstar_approx(dG)
    e = equilibrium(dG, float(dimer_18h.min()))
    print(f"{dG:>6.1f} {cs:>10.4g} {dimer_18h.min()/cs:>17.4g} "
          f"{e['capsid']/dimer_18h.min():>16.4f}")
print("\nEvery arm sits 10^2-10^4 above c*, so assembly is NOT concentration-limited")
print("in this system. The assembly module's leverage is kinetic - lag time and")
print("stalled fraction - not thermodynamic. Experiments should be designed accordingly:")
print("a titration meant to bracket c* would spend its whole range on one side of it.")

---
## 8 · Where the packaging and assembly arms meet

In MS2 the TR/pac hairpin binds the A/B dimer and stabilises the bent FG-loop conformer, so pac
binding does more than load cargo: it **lowers the nucleation barrier**. This gives a shared,
falsifiable prediction across the two arms, using the construct panel that already exists:

> The **0×pac** construct should show a longer nucleation lag and a lower capsid fraction at harvest
> than 3×pac, although both express the same coat protein.

The effect is modelled as extra effective contact energy on the first addition only. **This is a
qualitative coupling and not a calibrated one**; the magnitude is illustrative.

The result below does not support the prediction above, and it changes the experiment that should
be run. The two regimes should be compared before any experiment is designed.

In [ ]:
def run_with_nucleation_aid(dG_contact, dG_nuc_aid, **kw):
    '''pac binding stabilises the A/B conformer -> the first addition behaves as
    if it formed extra contact energy. Applied to step 2 only.'''
    sched = contact_schedule().copy()
    sched[2] += -dG_nuc_aid/abs(dG_contact)        # extra effective contacts
    return run_cascade(dG_contact, schedule=sched, **kw)

rows = []
for regime, kw, denom in [
        ("fed, in vivo (18 h)", dict(bolus=0.0, t_end=T_END,
                                     feed=lambda t: TOTAL/T_END), TOTAL),
        ("bolus 30 uM, in vitro", dict(bolus=30.0, t_end=T_END), 30.0)]:
    for lab, aid in [("0x pac", 0.0), ("1x pac", 0.8), ("3x pac", 1.8)]:
        r = run_with_nucleation_aid(DG_CONTACT, aid, **kw)
        cap = N_DIMERS*r.y[N_DIMERS]/denom
        t_half = r.t[np.argmax(cap > 0.5*cap[-1])]/60 if cap[-1] > 1e-6 else np.nan
        rows.append(dict(regime=regime, construct=lab, dG_nuc_aid=-aid,
                         capsid_frac=cap[-1], t_to_half_h=t_half))
pac = pd.DataFrame(rows); display(pac.round(4))

fed = pac[pac.regime.str.startswith("fed")].capsid_frac
vit = pac[pac.regime.str.startswith("bolus")].capsid_frac
print(f"fed regime   : capsid fraction spans {fed.min():.4f} - {fed.max():.4f} "
      f"({100*(fed.max()-fed.min()):.1f} percentage points)")
print(f"bolus regime : capsid fraction spans {vit.min():.4f} - {vit.max():.4f} "
      f"({100*(vit.max()-vit.min()):.1f} percentage points)")

> **The prediction does not hold in the fed regime.** With continuous feed the free-dimer pool stays
> far above what nucleation requires, so nucleation is not rate-limiting and the pac aid changes
> essentially nothing by 18 h. Comparing harvested VLP yield between 3×pac and 0×pac
> would therefore **not** test the allosteric coupling, and a null result there would be
> uninformative.
>
> The effect is visible only where nucleation *is* limiting: at low dimer concentration, that is, in
> **in vitro reassembly at ~20–50 µM** or in the first hours after induction. The informative
> experiment is therefore purified dimer ± TR hairpin, followed by SEC or light scattering. This
> differs from the experiment proposed at the start of the section; the model is what changed the
> design.

---
## 9 · What survives the placeholders?

The contact schedule and the degeneracy are the weakest part of this module. The scan below varies
both and reports which conclusions change. Conclusions that vary widely need the structural contact
count before they are reported; those that do not vary can be stated now.

In [ ]:
rows = []
for c_end in (2.02, 3.0, 4.0):                   # flat, moderate ramp, steep ramp
    for f_deg in (1.0, 2.0, 4.0):
        sch = contact_schedule(c_end=c_end)
        e = equilibrium(DG_CONTACT, 500.0, f_deg=f_deg, schedule=sch)
        r = run_cascade(DG_CONTACT, bolus=500.0, f_deg=f_deg, schedule=sch)
        cap = N_DIMERS*r.y[N_DIMERS, -1]/500.0
        rows.append(dict(c_end=c_end, f_deg=f_deg, cstar_uM=cstar_approx(DG_CONTACT),
                         eq_capsid=e["capsid"]/500.0, kinetic_capsid_18h=cap,
                         stalled=(500.0-N_DIMERS*r.y[N_DIMERS,-1]-r.y[1,-1])/500.0))
sens = pd.DataFrame(rows); display(sens.round(4))
print(f"c*        : identical across all 9 - it depends only on the TOTAL contact")
print(f"            count, which IS constrained. Safe to report.")
print(f"eq capsid : range {sens.eq_capsid.min():.4f} - {sens.eq_capsid.max():.4f}. Robust.")
print(f"stalled   : range {sens.stalled.min():.4f} - {sens.stalled.max():.4f}. NOT robust -")
print(f"            do not quote a stalled fraction until the schedule comes from")
print(f"            the structure.")

---
## 10 · Export and next steps

**Can be stated now:** $c^{*}$ and the equilibrium capsid fraction, since both depend only on the
total contact count. The conclusion that every expression arm lies far above $c^{*}$, so that
assembly is limited by kinetics and not by thermodynamics, is also robust.

**Cannot be stated until the contact schedule comes from the structure:** the stalled-intermediate
fraction, the nucleation lag time, and anything derived from the intermediate size distribution.

**Experiments supported by this module:**

- **Disassembly midpoint (urea or thermal ramp + SEC).** This is the most direct constraint on
  $\Delta G_{contact}$ available and is independent of the dPCR arm. It is inverted as described in
  Section 5.1 of the framework document.
- **SEC or DLS on the induction titration**, looking for a shoulder between the dimer and capsid
  peaks. The shoulder is the stalled population, and its presence or absence is the clearest test of
  §5–6.
- **In vitro reassembly at 20–50 µM dimer ± TR hairpin**, by SEC or light scattering (§8). A
  harvested-yield comparison is not informative: at in vivo dimer levels nucleation is not
  rate-limiting and the allosteric effect cannot be seen.
- **A titration designed to bracket $c^{*}$ is not useful**: §7 shows that the entire accessible
  range lies $10^2$–$10^4$ above it.

**Open modelling gap:** the model has no off-pathway or malformed-particle species. If TEM shows
malformed particles increasing with induction, that pathway needs to be added, and the current
yield predictions are optimistic.
---
## 11 · References

1. Zlotnick A. To build a virus capsid: an equilibrium model of the self assembly of polyhedral protein complexes. *J Mol Biol.* 1994;241(1):59–67.
2. Endres D, Zlotnick A. Model-based analysis of assembly kinetics for virus capsids or other spherical polymers. *Biophys J.* 2002;83(2):1217–1230.
3. Zlotnick A, Johnson JM, Wingfield PW, Stahl SJ, Endres D. A theoretical model successfully identifies features of hepatitis B virus capsid assembly. *Biochemistry.* 1999;38(44):14644–14652.
4. Hagan MF, Elrad OM. Understanding the concentration dependence of viral capsid assembly kinetics — the origin of the kinetically hindered regime. *Biophys J.* 2010;98(6):1065–1074.
5. Golmohammadi R, Valegård K, Fridborg K, Liljas L. The refined structure of bacteriophage MS2 at 2.8 Å resolution. *J Mol Biol.* 1993;234(3):620–639.
6. Valegård K, Murray JB, Stonehouse NJ, van den Worm S, Stockley PG, Liljas L. The three-dimensional structures of two complexes between recombinant MS2 capsids and RNA operator fragments. *J Mol Biol.* 1997;270(5):724–738.
7. Dykeman EC, Stockley PG, Twarock R. Solving a Levinthal's paradox for virus assembly identifies a unique antiviral strategy. *PNAS.* 2014;111(14):5361–5366.
8. Stockley PG, Rolfsson O, Thompson GS, et al. A simple, RNA-mediated allosteric switch controls the pathway to formation of a T=3 viral capsid. *J Mol Biol.* 2007;369(2):541–552.
9. Basnak G, Morton VL, Rolfsson Ó, Stonehouse NJ, Ashcroft AE, Stockley PG. Viral genomic single-stranded RNA directs the pathway toward a T=3 capsid. *J Mol Biol.* 2010;395(5):924–936.
10. Hagan MF. Modeling viral capsid assembly. *Adv Chem Phys.* 2014;155:1–68.

In [ ]:
np.savez_compressed("M3_export_literature.npz",
                    N_dimers=N_DIMERS, n_contacts=N_CONTACTS,
                    contact_schedule=contact_schedule(),
                    equilibrium_table=eq.to_numpy(),
                    equilibrium_cols=np.array(eq.columns),
                    sensitivity=sens.to_numpy(),
                    sensitivity_cols=np.array(sens.columns),
                    dimer_pool_uM=dimer_18h)
print("wrote M3_export_literature.npz")

## 12 · What the measured melting point changes

Every number above was obtained with a contact energy inverted from a published denaturation midpoint and not assumed. The panel below sweeps the whole range allowed by the thermal-shift assay, in which the midpoint is read at a coat-protein concentration quoted as a 5–20 µM monomer range, corresponding to −4.05 … −3.59 kcal mol⁻¹. It shows which module outputs survive that spread and which do not.

In [ ]:
DG_BAND = _lit[_lit.variant == "wtMS2"].sort_values("assay_dimer_uM")
dg_grid = np.linspace(-4.35, -3.45, 19)
cst, eqf, kin = [], [], []
for dg in dg_grid:
    cst.append(cstar_approx(dg))
    eqf.append(equilibrium(dg, 500.0)["capsid"]/500.0)
    r = run_cascade(dg, bolus=500.0)
    kin.append(N_DIMERS*r.y[N_DIMERS, -1]/500.0)
cst, eqf, kin = map(np.asarray, (cst, eqf, kin))

fig, axs = plt.subplots(1, 2, figsize=(10.2, 3.6))
lo, hi = DG_BAND.dG_contact_kcal_mol.min(), DG_BAND.dG_contact_kcal_mol.max()
for a in axs:
    a.axvspan(lo, hi, color=PALE, alpha=.55, lw=0, zorder=0,
              label="assay concentration range" if a is axs[0] else None)
    a.axvline(DG_CONTACT, color=NAVY, lw=1.2, ls="--",
              label=f"measured, {DG_CONTACT:.2f}" if a is axs[0] else None)
    a.axvline(-4.2, color=STEEL, lw=1.2, ls=":",
              label="placeholder, $-$4.20" if a is axs[0] else None)
    a.set_xlabel("$\\Delta G_{contact}$ (kcal mol$^{-1}$)")
axs[0].semilogy(dg_grid, cst, color=BLUE, lw=2)
axs[0].axhline(dimer_18h.min(), color=MID, lw=1, ls="-.")
axs[0].text(-4.33, dimer_18h.min()*1.25, "lowest M2 dimer arm", fontsize=6.5, color=MID)
axs[0].set(ylabel="$c^{*}$ ($\\mu$M)", title="Critical concentration")
axs[0].legend(fontsize=6.5, loc="lower right")
axs[1].plot(dg_grid, eqf, color=NAVY, lw=2, label="equilibrium")
axs[1].plot(dg_grid, kin, color=SKY, lw=2, label="kinetic, 18 h, 500 $\\mu$M bolus")
axs[1].set(ylabel="capsid fraction", ylim=(0, 1.05), title="Thermodynamics vs kinetics")
axs[1].legend(fontsize=6.5, loc="lower left")
fig.tight_layout(); plt.show()

i_lo, i_hi = int(np.argmin(abs(dg_grid-lo))), int(np.argmin(abs(dg_grid-hi)))
print(f"across the assay-concentration band ({hi:.2f} to {lo:.2f} kcal/mol):")
print(f"  c*                 {cst[i_hi]:.2f} -> {cst[i_lo]:.2f} uM   ({cst[i_hi]/cst[i_lo]:.1f}x)")
print(f"  equilibrium capsid {eqf[i_hi]:.4f} -> {eqf[i_lo]:.4f}   ({100*abs(eqf[i_hi]-eqf[i_lo]):.2f} pp)")
print(f"  kinetic capsid 18h {kin[i_hi]:.4f} -> {kin[i_lo]:.4f}   ({100*abs(kin[i_hi]-kin[i_lo]):.1f} pp)")
j = int(np.argmin(abs(dg_grid+4.2)))
print(f"placeholder -4.20 vs measured {DG_CONTACT:.2f}: c* {cst[j]:.2f} -> "
      f"{cst[int(np.argmin(abs(dg_grid-DG_CONTACT)))]:.2f} uM, kinetic capsid "
      f"{kin[j]:.3f} -> {kin[int(np.argmin(abs(dg_grid-DG_CONTACT)))]:.3f}")